In [19]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import os
import json
from tqdm.notebook import tqdm  # Use tqdm.notebook for nice progress bars in Jupyter
import string
from itertools import combinations
from collections import Counter
import time
import requests
import collections
import matplotlib.pyplot as plt
import random


MAX_LENGTH = 20

MODEL_SAVE_PATH = "../models/final_model.pt"
TOKENIZER_PATH = "../data/tokenizer.json"
WORDS_PATH = "../data/words_250000_train.txt"
OUTPUT_DATASET_PATH = "../data/big_dataset_self_play.csv"

print(f"Using device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=MAX_LENGTH):
        super(PositionalEncoding, self).__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]
    
class HangmanTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=4, num_layers=4, dim_feedforward=512, dropout=0.1):
        super(HangmanTransformer, self).__init__()

        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                                   dim_feedforward=dim_feedforward,
                                                   dropout=dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.wrong_guesses_proj = nn.Linear(26, d_model)
        self.pooling = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 26)
        )

    def forward(self, input_ids, attention_mask, wrong_guesses):
        embedded = self.token_embedding(input_ids)
        embedded = self.positional_encoding(embedded)
        encoded = self.transformer_encoder(embedded, src_key_padding_mask=~attention_mask.bool())
        encoded_transposed = encoded.transpose(1, 2)
        pooled = self.pooling(encoded_transposed).squeeze(-1)
        wrong_guesses_embedding = self.wrong_guesses_proj(wrong_guesses)
        combined = torch.cat([pooled, wrong_guesses_embedding], dim=1)
        logits = self.classifier(combined)
        return logits


print("Loading tokenizer...")
with open(TOKENIZER_PATH, "r") as f:
    tokenizer_dict = json.load(f)
id_to_token = {v: k for k, v in tokenizer_dict.items()}

print(f"Tokenizer loaded with {len(tokenizer_dict)} tokens")

print("Loading model...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

checkpoint = torch.load(MODEL_SAVE_PATH, map_location=device)
model = HangmanTransformer(vocab_size=checkpoint['vocab_size'])
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()

print("Model loaded successfully")


def encode(text, max_length=MAX_LENGTH):
    if not isinstance(text, str):
        text = str(text)
    token_ids = [tokenizer_dict.get(char, tokenizer_dict.get("<unk>", 0)) for char in text]
    if len(token_ids) < max_length:
        token_ids = token_ids + [tokenizer_dict.get("<pad>", 0)] * (max_length - len(token_ids))
    else:
        token_ids = token_ids[:max_length]
    return token_ids


def build_dictionary(dictionary_file_location):
        text_file = open(dictionary_file_location,"r")
        full_dictionary = text_file.read().splitlines()
        text_file.close()
        return full_dictionary


from nltk.corpus import words as nltk_words

def filter_english_words(word_list):
    english_words = set(w.lower() for w in nltk_words.words())
    
    filtered_words = [word for word in word_list if word.lower() in english_words]
    
    print(f"Original corpus size: {len(word_list)}")
    print(f"Filtered corpus size: {len(filtered_words)}")
    
    return filtered_words



def make_model_wrong_guesses(masked_word: str, word: str, wrong_k: int):
    """
    Run the model on (masked_word, no wrong_guesses yet),
    take its sigmoid-probabilities, and pick the top wrong_k
    letters that are NOT in the true word.
    """

    input_ids = torch.tensor([encode(masked_word)]).to(device)

    pad_id = tokenizer_dict.get("<pad>", 0)
    attn = (input_ids != pad_id).long()

    wg_vec = torch.zeros(1, 26, device=device)

    with torch.no_grad():
        logits = model(input_ids, attn, wg_vec)            
        probs  = torch.sigmoid(logits).squeeze(0)        


    _, idxs = torch.sort(probs, descending=True)
    alphabet = list('abcdefghijklmnopqrstuvwxyz')

    wrong_guesses = []
    for idx in idxs:
        letter = alphabet[idx.item()]

        if letter not in set(word):
            wrong_guesses.append(letter)
        if len(wrong_guesses) >= wrong_k:
            break

    return wrong_guesses

def generate_game_state(word: str, scenario: str):
    """
    scenario ∈ {'start','middle','end'} controls how many letters
    we mask (30/60/80% of distinct letters) and how many wrong
    guesses we draw from the model (1–2, 3–4, 5–6).
    """
    distinct = sorted(set(word))
    n = len(distinct)


    if scenario == 'start':
        mask_k  = max(1, int(0.3 * n))
        wrong_k = random.randint(1, 2)
    elif scenario == 'middle':
        mask_k  = max(1, int(0.6 * n))
        wrong_k = random.randint(3, 4)
    else: 
        mask_k  = max(1, int(0.8 * n))
        wrong_k = random.randint(4, 5)

    masked_letters = set(random.sample(distinct, mask_k))
    masked_word = "".join(ch if ch not in masked_letters else "_" for ch in word)

    wrong_guesses = make_model_wrong_guesses(masked_word, word, wrong_k)

    counts = Counter(word)
    target  = max(masked_letters, key=lambda c: counts[c])

    return masked_word, target, wrong_guesses

def create_finetuning_dataset(total_samples=10_000):
    lengths   = sorted(words_by_length.keys())
    per_len   = total_samples // len(lengths)
    remainder = total_samples % len(lengths)
    data      = []
    scenarios = ['start','middle','end']

    for i, L in enumerate(lengths):
        count = per_len + (1 if i < remainder else 0)
        pool  = words_by_length[L]
        chosen = random.sample(pool, k=min(count, len(pool)))

        for w in chosen:
            scenario = random.choice(scenarios)
            mword, tgt, wrongs = generate_game_state(w, scenario)
            data.append((mword, tgt, w, wrongs))
        
        print('processed words of lenth', L)

    return data


full_dictionary = build_dictionary(WORDS_PATH)
english_lines = filter_english_words(full_dictionary)

words_by_length = {}
for w in english_lines:
    L = len(w)
    if 3 <= L <= MAX_LENGTH:
        words_by_length.setdefault(L, []).append(w)


Using device: cpu
Loading tokenizer...
Tokenizer loaded with 127 tokens
Loading model...
Using device: cpu
Model loaded successfully
Original corpus size: 227300
Filtered corpus size: 117157


In [20]:
random.seed(42)
samples = create_finetuning_dataset(50_000)
df = pd.DataFrame(samples, columns=['masked_word','target','original_word','wrong_guesses'])
df.to_csv("../data/big_finetune_dataset_english_only.csv", index=False)
print("Saved finetuning dataset with", len(df), "rows.")

/Users/apple/Library/Python/3.9/lib/python/site-packages/torch/nn/modules/transformer.py:384: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/NestedTensorImpl.cpp:179.)
  output = torch._nested_tensor_from_mask(output, src_key_padding_mask.logical_not(), mask_check=False)


processed words of lenth 3
processed words of lenth 4
processed words of lenth 5
processed words of lenth 6
processed words of lenth 7
processed words of lenth 8
processed words of lenth 9
processed words of lenth 10
processed words of lenth 11
processed words of lenth 12
processed words of lenth 13
processed words of lenth 14
processed words of lenth 15
processed words of lenth 16
processed words of lenth 17
processed words of lenth 18
processed words of lenth 19
processed words of lenth 20
Saved finetuning dataset with 36999 rows.


In [22]:
import os
import json
import ast
import random
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import math
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import os
import json
from tqdm.notebook import tqdm  # Use tqdm.notebook for nice progress bars in Jupyter
import string
from itertools import combinations
from collections import Counter
import time
import requests
import collections
import matplotlib.pyplot as plt
import random


MAX_LENGTH      = 20
TOKENIZER_PATH  = "../data/tokenizer.json"
MODEL_SAVE_PATH = "../models/final_model_finetuned.pt"
LOAD_PATH       = "../models/final_model.pt"
FINETUNE_CSV    = "../data/finetune_dataset_english_only.csv"
BATCH_SIZE      = 32
NUM_EPOCHS      = 40
LR              = 1e-3
DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ALPHABET        = list("abcdefghijklmnopqrstuvwxyz")

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=MAX_LENGTH):
        super(PositionalEncoding, self).__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]
    
class HangmanTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=4, num_layers=4, dim_feedforward=512, dropout=0.1):
        super(HangmanTransformer, self).__init__()

        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                                   dim_feedforward=dim_feedforward,
                                                   dropout=dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.wrong_guesses_proj = nn.Linear(26, d_model)
        self.pooling = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 26)
        )

    def forward(self, input_ids, attention_mask, wrong_guesses):
        embedded = self.token_embedding(input_ids)
        embedded = self.positional_encoding(embedded)
        encoded = self.transformer_encoder(embedded, src_key_padding_mask=~attention_mask.bool())
        encoded_transposed = encoded.transpose(1, 2)
        pooled = self.pooling(encoded_transposed).squeeze(-1)
        wrong_guesses_embedding = self.wrong_guesses_proj(wrong_guesses)
        combined = torch.cat([pooled, wrong_guesses_embedding], dim=1)
        logits = self.classifier(combined)
        return logits



class HangmanDataset(Dataset):
    def __init__(self, csv_path, tokenizer_dict, max_length=MAX_LENGTH):
        self.df = pd.read_csv(csv_path)
        self.tok = tokenizer_dict
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        masked     = row["masked_word"]
        target_chr = row["target"]
        wrongs     = ast.literal_eval(row["wrong_guesses"])

    
        ids = [ self.tok.get(ch, self.tok.get("<unk>",0)) for ch in masked ]
        pad_id = self.tok.get("<pad>",0)
        if len(ids) < self.max_length:
            ids += [pad_id] * (self.max_length - len(ids))
        else:
            ids = ids[:self.max_length]

        input_ids      = torch.tensor(ids, dtype=torch.long)
        attention_mask = (input_ids != pad_id).long()


        wg = torch.zeros(26, dtype=torch.float)
        for c in wrongs:
            if c in ALPHABET:
                wg[ALPHABET.index(c)] = 1.0

        tgt_idx = torch.tensor(ALPHABET.index(target_chr), dtype=torch.long)
        return input_ids, attention_mask, wg, tgt_idx

def collate_fn(batch):
    ids, masks, wgs, tgts = zip(*batch)
    return (
      torch.stack(ids), 
      torch.stack(masks), 
      torch.stack(wgs), 
      torch.stack(tgts),
    )


class LoRALinear(nn.Module):
    def __init__(self, orig: nn.Linear, r=8, alpha=32, dropout=0.05):
        super().__init__()
        self.orig = orig           
        self.r = r
        self.scaling = alpha / r if r > 0 else 0.0

        if r > 0:
            self.lora_A = nn.Linear(orig.in_features, r, bias=False)
            self.lora_B = nn.Linear(r, orig.out_features, bias=False)
            nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
            nn.init.zeros_(self.lora_B.weight)
            self.lora_dropout = nn.Dropout(dropout)
        else:
            self.lora_A = self.lora_B = None

        self.weight = orig.weight
        self.bias   = orig.bias

    def forward(self, x):
        out = self.orig(x)
        if self.r > 0 and self.training:
            delta = self.lora_B(self.lora_dropout(self.lora_A(x))) * self.scaling
            return out + delta
        return out

def inject_lora(model: nn.Module, r=8, alpha=16, skip=["token_embedding"], dropout=0.05):
    for name, child in list(model.named_children()):
        if any(s in name for s in skip):
            continue
        if isinstance(child, nn.Linear):
            setattr(model, name, LoRALinear(child, r=r, alpha=alpha, dropout=dropout))
        else:
            inject_lora(child, r=r, alpha=alpha, skip=skip, dropout=dropout)

def freeze_base(model: nn.Module):
    for n, p in model.named_parameters():
        p.requires_grad = ("lora_A" in n) or ("lora_B" in n)

with open(TOKENIZER_PATH) as f:
    tokenizer_dict = json.load(f)

checkpoint = torch.load(LOAD_PATH, map_location=DEVICE)
model = HangmanTransformer(vocab_size=checkpoint["vocab_size"])
model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)

inject_lora(model, r=16, alpha=64, skip=["token_embedding", "classifier"], dropout=0.05)
freeze_base(model)
for p in model.classifier.parameters():
    p.requires_grad = True

for name, param in model.named_parameters():
    if "transformer_encoder.layers.3" in name:
        param.requires_grad = True   

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"🔍 Training {trainable}/{total} params ({100*trainable/total:.2f}%)")


ds     = HangmanDataset(FINETUNE_CSV, tokenizer_dict)
loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), 
    lr=LR, weight_decay=1e-2
)
criterion = nn.CrossEntropyLoss()

for epoch in range(1, NUM_EPOCHS+1):
    model.train()
    tot_loss = 0
    for input_ids, attn_mask, wrongs, targets in loader:
        input_ids   = input_ids.to(DEVICE)
        attn_mask   = attn_mask.to(DEVICE)
        wrongs      = wrongs.to(DEVICE)
        targets     = targets.to(DEVICE)

        logits = model(input_ids=input_ids,
                       attention_mask=attn_mask,
                       wrong_guesses=wrongs)
        loss = criterion(logits, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        tot_loss += loss.item() * input_ids.size(0)

    print(f"Epoch {epoch:02d} — avg loss: {tot_loss/len(ds):.4f}")

🔍 Training 800698/2421434 params (33.07%)
Epoch 01 — avg loss: 1.9273
Epoch 02 — avg loss: 1.6385
Epoch 03 — avg loss: 1.5163
Epoch 04 — avg loss: 1.3989
Epoch 05 — avg loss: 1.3088
Epoch 06 — avg loss: 1.2169
Epoch 07 — avg loss: 1.1536
Epoch 08 — avg loss: 1.0784
Epoch 09 — avg loss: 1.0157
Epoch 10 — avg loss: 0.9393
Epoch 11 — avg loss: 0.9032
Epoch 12 — avg loss: 0.8630
Epoch 13 — avg loss: 0.8209
Epoch 14 — avg loss: 0.7652
Epoch 15 — avg loss: 0.7573
Epoch 16 — avg loss: 0.7271
Epoch 17 — avg loss: 0.6944
Epoch 18 — avg loss: 0.6783
Epoch 19 — avg loss: 0.6108
Epoch 20 — avg loss: 0.6217
Epoch 21 — avg loss: 0.5736
Epoch 22 — avg loss: 0.5670
Epoch 23 — avg loss: 0.5553
Epoch 24 — avg loss: 0.5375
Epoch 25 — avg loss: 0.5115
Epoch 26 — avg loss: 0.4975
Epoch 27 — avg loss: 0.4907
Epoch 28 — avg loss: 0.4607
Epoch 29 — avg loss: 0.4764
Epoch 30 — avg loss: 0.4672
Epoch 31 — avg loss: 0.4745
Epoch 32 — avg loss: 0.4176
Epoch 33 — avg loss: 0.4480
Epoch 34 — avg loss: 0.4341
Epoch 

In [85]:
torch.save(model.state_dict(), os.path.join(MODEL_SAVE_PATH))